**Fine-Tuning the DistilBERT based Uncased Model**

Connect to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Install Package

In [ ]:
!pip install -q -U trl transformers accelerate git+https://github.com/huggingface/peft.git
!pip install -q datasets bitsandbytes einops wandb

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.6/316.6 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 330.9/330.9 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.8/109.8 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.7/472.7 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.4/122.4 MB 8.5 

Import Library

In [ ]:
from datasets import load_dataset, Dataset
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, BitsAndBytesConfig
from transformers import Trainer, TrainingArguments
import pandas as pd
import torch
import os

Dataset

In [ ]:
# Train Dataset
train_dataset = load_dataset('fancyzhx/ag_news', split="train")
train_dataset = pd.DataFrame(train_dataset)
train_dataset = Dataset.from_pandas(train_dataset)

# Test Dataset
test_dataset = load_dataset('fancyzhx/ag_news', split="test")
test_dataset = pd.DataFrame(test_dataset)
test_dataset = Dataset.from_pandas(test_dataset)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

Labels

In [ ]:
labels = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "Sci/Tech"
}

Model (Before Fine-Tune)

In [ ]:
model_name = 'distilbert-base-uncased'

# quantization_config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

model = DistilBertForSequenceClassification.from_pretrained(model_name,quantization_config=bnb_config, num_labels=4)
based_model = DistilBertForSequenceClassification.from_pretrained(model_name, num_labels=4)
model.config.use_cache = False

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

`low_cpu_mem_usage` was None, now default to True since model is quantized.


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
print(model)

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear4bit(in_features=768, out_features=768, bias=True)
            (k_lin): Linear4bit(in_features=768, out_features=768, bias=True)
            (v_lin): Linear4bit(in_features=768, out_features=768, bias=True)
            (out_lin): Linear4bit(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1,

In [ ]:
print(based_model)

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


Tokenization

In [ ]:
# Preprocessing
tokenizer = DistilBertTokenizer.from_pretrained(model_name)

# Tokenize the dataset
def preprocess_function(examples):
    tokenized = tokenizer(examples['text'], truncation=True, padding='max_length', max_length=512)
    tokenized['labels'] = examples['label']  # Ensure labels are added as 'labels'
    return tokenized

# Preprocess the train dataset
tokenized_train_dataset = train_dataset.map(preprocess_function, batched=True, remove_columns=train_dataset.column_names)
tokenized_train_dataset.set_format('torch')

# Preprocess the test dataset
tokenized_test_dataset = test_dataset.map(preprocess_function, batched=True, remove_columns=test_dataset.column_names)
tokenized_test_dataset.set_format('torch')

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [ ]:
# Calculate the total number of parameters in the model
num_params = sum(p.numel() for p in model.parameters())

# Calculate the approximate size of the model in MB
# Assuming each parameter uses 4 bytes (float32)
model_size_approx = num_params * 4 / (1024 * 1024)  # Size in MB
print(f"Approximate model size before fine-tuning: {model_size_approx:.2f} MB")

Approximate model size before fine-tuning: 173.29 MB


In [ ]:
# Calculate the total number of parameters in the model
num_params = sum(p.numel() for p in based_model.parameters())

# Calculate the approximate size of the model in MB
# Assuming each parameter uses 4 bytes (float32)
model_size_approx = num_params * 4 / (1024 * 1024)  # Size in MB
print(f"Approximate model size before fine-tuning: {model_size_approx:.2f} MB")

Approximate model size before fine-tuning: 255.42 MB


Set training data batch size

In [ ]:
per_device_train_batch_size = 16
gradient_accumulation_steps = 2
total_train_data_size = len(train_dataset)  # Total training data size
batch_size = per_device_train_batch_size * 1  # Assuming 1 GPU
steps_per_epoch = int(total_train_data_size / (batch_size * gradient_accumulation_steps))
desired_epochs = 1  # Example of desired number of epochs

# Calculate max_steps for TrainingArguments
max_steps_calculated = steps_per_epoch * desired_epochs

Accuracy before fine-tuning

In [ ]:
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

In [ ]:
# Set model to evaluation mode
model.eval()

# Store results
predictions = []
true_labels = []

# Model testing
for batch in DataLoader(tokenized_test_dataset, batch_size=per_device_train_batch_size):
    with torch.no_grad():
        outputs = model(**batch)

    logits = outputs.logits
    predicted_labels = torch.argmax(logits, dim=-1).tolist()

    predictions.extend(predicted_labels)
    true_labels.extend(batch['labels'].tolist())

# Calculate accuracy
accuracy_before_finetune = accuracy_score(true_labels, predictions)
print(f"Accuracy before fine-tuning: {accuracy_before_finetune:.4f}")

# Generate classification report
report = classification_report(true_labels, predictions, target_names=list(labels.values()))
print(report)

Accuracy before fine-tuning: 0.2399
              precision    recall  f1-score   support

       World       0.27      0.39      0.32      1900
      Sports       0.11      0.02      0.03      1900
    Business       0.00      0.00      0.00      1900
    Sci/Tech       0.23      0.55      0.33      1900

    accuracy                           0.24      7600
   macro avg       0.15      0.24      0.17      7600
weighted avg       0.15      0.24      0.17      7600



Fine-Tuning

In [ ]:
from peft import LoraConfig

lora_alpha = 16
lora_dropout = 0.1
lora_r = 64

peft_config = LoraConfig(
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    r=lora_r,
    bias="none",
    task_type="SEQ_CLASS_LM", #Fix
    target_modules=[
        "attention.q_lin",
        "attention.k_lin",
        "attention.v_lin"    ]
)

In [ ]:
output_dir = "/content/drive/MyDrive/llm_project_uncased/"
os.makedirs(output_dir, exist_ok=True)
gradient_accumulation_steps = gradient_accumulation_steps
optim = "paged_adamw_32bit"
save_steps = 1000
logging_steps = 10
learning_rate = 2e-4
max_grad_norm = 0.3
max_steps = 100
warmup_ratio = 0.03
lr_scheduler_type = "cosine"

training_arguments = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    optim=optim,
    save_steps=save_steps,
    logging_steps=logging_steps,
    learning_rate=learning_rate,
    fp16=True,
    max_grad_norm=max_grad_norm,
    max_steps=max_steps_calculated,
    warmup_ratio=warmup_ratio,
    group_by_length=True,
    lr_scheduler_type=lr_scheduler_type,
)

In [ ]:
from trl import SFTTrainer
from transformers import DefaultDataCollator

max_seq_length = 512

def formatting_func(example):
    return example["text"]

data_collator = DefaultDataCollator()

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_test_dataset,
    peft_config=peft_config,
    dataset_text_field="input_ids",
    max_seq_length=max_seq_length,
    tokenizer=tokenizer,
    args=training_arguments,
    data_collator=data_collator,
)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field, max_seq_length. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.10/dist-packages/trl/trainer/sft_trainer.py:283: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/trl/trainer/sft_trainer.py:321: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/trl/trainer/sft_trainer.py:401: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `SFTTrainer.__init__`. 

Trainer

In [ ]:
for name, module in trainer.model.named_modules():
    if "norm" in name:
        module = module.to(torch.float32)

In [ ]:
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend. Please refer to https://wandb.me/wandb-core for more information.


<IPython.core.display.Javascript object>

wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


Step,Training Loss
10,2.803500
20,2.795600
30,2.780900
40,2.794500
50,2.775300
60,2.780800
70,2.759200
80,2.746700
90,2.702500
100,2.612500


TrainOutput(global_step=3750, training_loss=0.7887677663167317, metrics={'train_runtime': 1636.3194, 'train_samples_per_second': 73.335, 'train_steps_per_second': 2.292, 'total_flos': 1.654895296512e+16, 'train_loss': 0.7887677663167317, 'epoch': 1.0})

In [ ]:
print(model)

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): lora.Linear4bit(
              (base_layer): Linear4bit(in_features=768, out_features=768, bias=True)
              (lora_dropout): ModuleDict(
                (default): Dropout(p=0.1, inplace=False)
              )
              (lora_A): ModuleDict(
                (default): Linear(in_features=768, out_features=64, bias=False)
              )
              (lora_B): ModuleDict(
                (default): Linear(in_features=64, out_featur

Accuracy after fine-tuning

In [ ]:
# Set model to evaluation mode
model.eval()

# Store results
predictions = []
true_labels = []

# Model testing
for batch in DataLoader(tokenized_test_dataset, batch_size=per_device_train_batch_size):
    with torch.no_grad():
        outputs = model(**batch)

    logits = outputs.logits
    predicted_labels = torch.argmax(logits, dim=-1).tolist()

    predictions.extend(predicted_labels)
    true_labels.extend(batch['labels'].tolist())

# Calculate accuracy
accuracy_after_finetune = accuracy_score(true_labels, predictions)
print(f"Accuracy after fine-tuning: {accuracy_after_finetune:.4f}")

# Generate classification report
report = classification_report(true_labels, predictions, target_names=list(labels.values()))
print(report)

Accuracy after fine-tuning: 0.9124
              precision    recall  f1-score   support

       World       0.94      0.89      0.91      1900
      Sports       0.96      0.99      0.97      1900
    Business       0.88      0.87      0.87      1900
    Sci/Tech       0.88      0.91      0.89      1900

    accuracy                           0.91      7600
   macro avg       0.91      0.91      0.91      7600
weighted avg       0.91      0.91      0.91      7600



Save Model

In [ ]:
# Define path for saving the model
save_directory = "/content/drive/MyDrive/llm_project_uncased/full_model"

# Create folder if it doesn't already exist
os.makedirs(save_directory, exist_ok=True)

# 1. Convert PEFT model to full model
model = trainer.model.merge_and_unload()

# 2. Save the full model
model.save_pretrained(save_directory)

# 3. Save the tokenizer
tokenizer.save_pretrained(save_directory)

# 4. Save the config
model.config.save_pretrained(save_directory)

print(f"Full model and tokenizer have been saved to {save_directory}")

# Test loading the model back
try:
    loaded_model = DistilBertForSequenceClassification.from_pretrained(save_directory)
    loaded_tokenizer = DistilBertTokenizer.from_pretrained(save_directory)
    print("Model loaded successfully!")

    # Test prediction
    test_text = "Your test text here"
    inputs = loaded_tokenizer(test_text, return_tensors="pt", truncation=True, max_length=512)
    outputs = loaded_model(**inputs)
    print("Prediction test successful!")

except Exception as e:
    print(f"An error occurred: {str(e)}")

/usr/local/lib/python3.10/dist-packages/peft/tuners/lora/bnb.py:340: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(
Unused kwargs: ['_load_in_4bit', '_load_in_8bit', 'quant_method']. These kwargs are not used in <class 'transformers.utils.quantization_config.BitsAndBytesConfig'>.
`low_cpu_mem_usage` was None, now default to True since model is quantized.


Full model and tokenizer have been saved to /content/drive/MyDrive/llm_project_uncased/full_model
Model loaded successfully!
Prediction test successful!


Check Size of Model

In [ ]:
# คำนวณจำนวนพารามิเตอร์ทั้งหมดในโมเดล
num_params = sum(p.numel() for p in model.parameters())

# คำนวณขนาดประมาณของโมเดลในหน่วย MB
# โดยพิจารณาแต่ละพารามิเตอร์ใช้ 4 bytes (float32)
model_size_approx = num_params * 4 / (1024 * 1024)  # ขนาดใน MB
print(f"Approximate model size: {model_size_approx:.2f} MB")

Approximate model size: 173.29 MB
